# 01 · Data preparation (Person 1)
GENKI-4K and FER-2013: indexing, face cropping, normalization, quality checks, de-duplication and train/val/test splits.

All logic lives in `src/data/`; this notebook runs it and shows the results used in the report's *Dataset* and *Preprocessing* sections.
Re-running it regenerates `data/splits/*.csv`, `data/processed/*.npz` and `results/data/*`.

## 0 · Get the data (skip if `data/raw/` is already filled)
```bash
kaggle datasets download -d talhasar/genki4k -p data/raw/genki4k --unzip
kaggle datasets download -d msambare/fer2013 -p data/raw/fer2013 --unzip
```
In Colab prefix with `!` after uploading `kaggle.json`. Extra folder levels from unzipping are fine.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option("display.width", 140)

In [ ]:
from src.data.config import GENKI_RAW, FER_RAW, IMG_SIZE, FACE_MARGIN, GENKI_SPLIT, SEED
print("GENKI raw:", GENKI_RAW.exists(), "| FER raw:", FER_RAW.exists())
print(f"crop size {IMG_SIZE}x{IMG_SIZE}, face margin {FACE_MARGIN:.0%}, GENKI split {GENKI_SPLIT}, seed {SEED}")

## 1 · Check the download
Compares counts with the published sizes and confirms the official FER split is recovered from the Kaggle file names.

In [ ]:
from src.data import check_raw
assert check_raw.main() == 0, "fix the download before going on"

## 2 · Run the pipeline
Face detection on 4,000 GENKI photos takes about a minute; FER is pre-cropped and takes seconds.

In [ ]:
from src.data import prepare
prepare.main([])
summary = json.loads(Path("results/data/summary.json").read_text())

## 3 · What was kept and what was removed

In [ ]:
rows = []
for tag, s in summary.items():
    rows.append({"dataset": tag, "images": s["n_images"], "kept": s["n_kept"],
                 **{f"drop: {k}": v for k, v in s["drop_reasons"].items()},
                 **{f"{k} (kept)": v for k, v in s["split_sizes"].items()}})
pd.DataFrame(rows).set_index("dataset").fillna(0).astype(int).T

In [ ]:
for tag in summary:
    print(tag)
    display(pd.DataFrame(summary[tag]["raw_label_by_split"]).fillna(0).astype(int))
    display(pd.DataFrame(summary[tag]["game_label_by_split"])[["train","val","test"]])

## 4 · GENKI-4K: face detection, size and pose

In [ ]:
g = summary["genki4k"]
print("source:", g["source"], "| face method:", g["face_method"])
print("original size:", g["orig_size"])
if "abs_pose_deg" in g:
    print("photos with >1 face detected (largest kept):", g["n_faces_gt1"])
    display(pd.DataFrame(g["abs_pose_deg"]))

In [ ]:
display(Image("results/data/genki4k_pipeline.png"))
if Path("results/data/genki4k_pose.png").exists():
    display(Image("results/data/genki4k_pose.png"))

In [ ]:
display(Image("results/data/genki4k_class_balance.png"))
display(Image("results/data/genki4k_samples.png"))
display(Image("results/data/genki4k_dropped.png"))

## 5 · FER-2013: imbalance, duplicates and blank images

In [ ]:
display(Image("results/data/fer2013_class_balance.png"))
display(Image("results/data/fer2013_samples.png"))
display(Image("results/data/fer2013_dropped.png"))

## 6 · Sanity checks on the split files

In [ ]:
for tag in ["genki4k", "fer2013"]:
    df = pd.read_csv(f"data/splits/{tag}.csv")
    assert df["image_id"].is_unique
    k = df[df["keep"]]
    print(tag, "| rows", len(df), "| kept", len(k),
          "| expression rate by split:", k.groupby("split")["game_label"].mean().round(3).to_dict())

**Notes for the report**
- GENKI "non-smile" is not the same as neutral: some non-smile faces carry other expressions, so GENKI measures *smile detection*, not *neutrality*.
- GENKI has no subject IDs, so the random split cannot guarantee a person appears in only one split. Near-identical photos were removed across splits, but different photos of the same person may remain.
- FER-2013 keeps its official Usage split (Training / PublicTest / PrivateTest) so results are comparable with the literature.